# 07 · Envío al bucket del equipo

Sube un fichero de predicciones al bucket `prc-2026-knowledgeable-alligator` con `scripts/enviar.py`. El 00 no lo
ejecuta: se abre y se lanza a mano, y mientras `ENVIAR = 0` no sube nada.

- Con `PERIODO = "ranking"` (plantilla de 344.841 salidas) se sube como `knowledgeable-alligator_v<N>.parquet`, la
  siguiente versión libre.
- Con `PERIODO = "final"` (670.790 salidas) se sube como `knowledgeable-alligator_final.parquet`, el nombre que pide
  el reto. Si ya está en el bucket, solo se sustituye con `REEMPLAZAR = 1`.

Antes de subir comprueba el fichero contra la plantilla y que no se haya enviado ya (SHA-256), y guarda
`registro_envio.json` junto al fichero; después espera la puntuación hasta `ESPERA_MIN` minutos. El reto admite 5
envíos al día. Si cambias `PERIODO`, reinicia el kernel.

In [3]:
PERIODO = "final"     # "ranking" (ene + jul 2026) o "final" (ene, feb, jun y jul 2026)
ARCHIVO = "Submissions/v30_final_modelos/knowledgeable-alligator_v30_final_modelos.parquet"   # fichero a subir
NOTA = "v30 con los modelos guardados (sin reentrenar) sobre los cuatro meses finales"
ENVIAR = 1            # 1 = subir de verdad
REEMPLAZAR = 1        # solo final: 1 = sustituir un knowledgeable-alligator_final.parquet que ya esté en el bucket
ESPERA_MIN = 30       # minutos de espera de la puntuación

## 1. Validación y destino (no sube nada)

In [4]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "cuaderno.py").exists())
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT)]
from cuaderno import elegir_periodo  # noqa: E402

assert elegir_periodo(PERIODO) == PERIODO
import enviar as EV  # noqa: E402

reg = EV.preparar(ARCHIVO, NOTA, reemplazar=bool(REEMPLAZAR))
reg

{'periodo': 'final',
 'version': 'final',
 'objeto': 'knowledgeable-alligator_final.parquet',
 'reemplaza': False,
 'sha256': '03faee3d822ee45b38f264f8186b47f8abc514c15c2025dceac77e29b1cdd6db',
 'fichero_local': 'Submissions\\v30_final_modelos\\knowledgeable-alligator_v30_final_modelos.parquet',
 'filas': 670790,
 'nota': 'v30 con los modelos guardados (sin reentrenar) sobre los cuatro meses finales',
 'versiones_previas': [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  21,
  23,
  24,
  25,
  26,
  27,
  28,
  29]}

Comparación rápida con el v30 enviado, en las salidas que comparten ID (enero y julio).

In [5]:
import pandas as pd

p = pd.read_parquet(ROOT / ARCHIVO)
o = pd.read_parquet(ROOT / "Submissions" / "v30" / "knowledgeable-alligator_v30.parquet")
m = p.merge(o, on="MVT_ID_mvt", suffixes=("", "_v30"), validate="one_to_one")
dif = m.TAXITIME_SEC_mvt.astype(float) - m.TAXITIME_SEC_mvt_v30.astype(float)
print(f"{len(p):,} filas · media {p.TAXITIME_SEC_mvt.mean():.1f} s · {len(m):,} IDs comunes con v30: "
      f"diferencia media {dif.mean():+.1f} s, RMS {(dif ** 2).mean() ** 0.5:.1f} s")

670,790 filas · media 1014.0 s · 344,841 IDs comunes con v30: diferencia media -2.6 s, RMS 19.0 s


## 2. Subida

In [6]:
if ENVIAR:
    reg = EV.subir(reg, esperar_s=60 * ESPERA_MIN)
else:
    print(f"No se ha subido nada (ENVIAR = 0). Destino previsto: {reg['objeto']}")

subido knowledgeable-alligator_final.parquet
{
 "status": "Succeeded",
 "file": "knowledgeable-alligator_final.parquet",
 "bucket": "prc-2026-knowledgeable-alligator",
 "used_pairs": 670790,
 "inputs": {
  "test_set_bucket": "prc-2026-datasets",
  "test_set_file": "final_submitting.parquet"
 }
}


## 3. Puntuación (para consultarla más tarde)

In [8]:
from prc import data

data.resultado_envio(data.cliente_bucket(), reg["objeto"])

{'status': 'Succeeded',
 'file': 'knowledgeable-alligator_final.parquet',
 'bucket': 'prc-2026-knowledgeable-alligator',
 'used_pairs': 670790,
 'inputs': {'test_set_bucket': 'prc-2026-datasets',
  'test_set_file': 'final_submitting.parquet'}}